# A Guided Tour of the Tourism Redistribution Notebook

This notebook implements a decision support system (DSS) for managing overtourism in Japan. The central problem: millions of tourists crowd into the same handful of famous cities while dozens of equally compelling destinations sit underutilised. The system takes a tourist who is headed to a congested anchor (say, Kyoto) and recommends ranked alternatives from a pool of lesser-known orbits (Kanazawa, Tottori, etc.) based on preference matching, congestion signals, and travel feasibility. If you have worked with recommendation systems or multi-criteria scoring before, the architecture will feel familiar. If not, each section below explains the reasoning before the code runs.


## Loading the Toolbox

Standard imports: pandas, numpy, sklearn (TF-IDF, KNN, MinMaxScaler), networkx for graph traversal, and ipywidgets for the interactive dashboard. Also pulls in `io` and `contextlib` for stdout suppression during the calibration sweep later.


In [14]:
import pandas as pd
import numpy as np
import math
import io
import contextlib
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

import ipywidgets as widgets
from IPython.display import display, clear_output

print("All imports loaded successfully.")


All imports loaded successfully.


## The 30-Destination Dataset

This cell constructs a synthetic dataset of 30 Japanese destinations: 10 anchors (major cities like Tokyo, Kyoto, Osaka) and 20 orbits (smaller towns like Kanazawa, Takayama, Koyasan). Each entry carries tourist counts, capacity limits, delay metrics, quality scores (cultural, experiential, infrastructure, cleanliness), logistical flags, lat/lon coordinates, and a tag list describing the destination's character. The data is deliberately synthetic so we can control congestion patterns and test the scoring logic under known conditions.


In [15]:
# ── 30-Location Japan Tourism Dataset (Expanded Tags + Coordinates) ──────────
data = [
    # ═══════════════════════  MAJOR ANCHORS  ═══════════════════════
    {"region": "Tokyo", "is_anchor": True, "tourist_count": 200000, "capacity": 150000,
     "avg_delay_minutes": 25, "cultural_score": 85, "experiential_score": 95,
     "infrastructure_score": 95, "promotion_score": 100, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.6895, "lon": 139.6917,
     "tags": ["urban", "modern", "shopping", "food", "nightlife", "anime",
             "tech", "skyscraper", "street_food", "museum", "lively",
             "fashion", "train_hub", "neon", "pop_culture"]},

    {"region": "Kyoto", "is_anchor": True, "tourist_count": 120000, "capacity": 80000,
     "avg_delay_minutes": 30, "cultural_score": 95, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 95, "cleanliness_score": 72,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.0116, "lon": 135.7681,
     "tags": ["historic", "temple", "shrine", "cherry_blossom", "traditional",
             "unesco", "culture", "geisha", "gardens", "bamboo",
             "matcha", "photo_spot", "architecture", "autumn_leaves", "crafts"]},

    {"region": "Osaka", "is_anchor": True, "tourist_count": 150000, "capacity": 120000,
     "avg_delay_minutes": 20, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 90, "promotion_score": 90, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.6937, "lon": 135.5023,
     "tags": ["urban", "food", "nightlife", "modern", "street_food",
             "castle", "lively", "comedy", "shopping", "takoyaki",
             "neon", "canal", "entertainment"]},

    {"region": "Hakone", "is_anchor": True, "tourist_count": 60000, "capacity": 40000,
     "avg_delay_minutes": 35, "cultural_score": 85, "experiential_score": 90,
     "infrastructure_score": 80, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.2326, "lon": 139.1070,
     "tags": ["hot_spring", "nature", "mountain", "view", "lake",
             "ryokan", "cable_car", "volcanic", "serene", "art_museum",
             "scenic_train", "relaxation", "fuji_view"]},

    {"region": "Nara", "is_anchor": True, "tourist_count": 70000, "capacity": 50000,
     "avg_delay_minutes": 25, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 80, "promotion_score": 80, "cleanliness_score": 74,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 34.6851, "lon": 135.8048,
     "tags": ["historic", "temple", "nature", "deer", "unesco",
             "traditional", "park", "buddha", "ancient", "serene",
             "photo_spot", "culture", "architecture"]},

    {"region": "Hiroshima", "is_anchor": True, "tourist_count": 90000, "capacity": 70000,
     "avg_delay_minutes": 20, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 75, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.3853, "lon": 132.4553,
     "tags": ["historic", "memorial", "peace", "temple", "island",
             "unesco", "culture", "coastal", "tram", "okonomiyaki",
             "architecture", "museum", "solemn"]},

    {"region": "Kamakura", "is_anchor": True, "tourist_count": 55000, "capacity": 40000,
     "avg_delay_minutes": 28, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 80, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 35.3192, "lon": 139.5467,
     "tags": ["historic", "temple", "coastal", "buddha", "shrine",
             "hiking", "beach", "traditional", "serene", "photo_spot",
             "architecture", "crafts", "day_trip"]},

    {"region": "Nikko", "is_anchor": True, "tourist_count": 45000, "capacity": 35000,
     "avg_delay_minutes": 30, "cultural_score": 92, "experiential_score": 88,
     "infrastructure_score": 70, "promotion_score": 75, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 36.7500, "lon": 139.6000,
     "tags": ["historic", "shrine", "mountain", "nature", "unesco",
             "waterfall", "autumn_leaves", "ornate", "cedar", "lake",
             "hiking", "traditional", "architecture"]},

    {"region": "Sapporo", "is_anchor": True, "tourist_count": 80000, "capacity": 70000,
     "avg_delay_minutes": 15, "cultural_score": 75, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 85, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 43.0621, "lon": 141.3544,
     "tags": ["urban", "snow", "food", "festival", "beer",
             "ski", "ramen", "park", "modern", "winter_sport",
             "lively", "night_market", "scenic"]},

    {"region": "Fukuoka", "is_anchor": True, "tourist_count": 65000, "capacity": 60000,
     "avg_delay_minutes": 10, "cultural_score": 80, "experiential_score": 90,
     "infrastructure_score": 88, "promotion_score": 80, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.5904, "lon": 130.4017,
     "tags": ["urban", "food", "historic", "shopping", "ramen",
             "shrine", "night_market", "canal", "modern", "lively",
             "street_food", "port"]},

    # ═══════════════════════  ORBIT DESTINATIONS  ═══════════════════════
    {"region": "Kanazawa", "is_anchor": False, "tourist_count": 25000, "capacity": 50000,
     "avg_delay_minutes": 8, "cultural_score": 85, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.5613, "lon": 136.6562,
     "tags": ["historic", "cherry_blossom", "garden", "traditional", "crafts",
             "geisha", "samurai", "museum", "culture", "serene",
             "architecture", "matcha", "autumn_leaves"]},

    {"region": "Takayama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 80, "experiential_score": 75,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 90,
     "has_coach_parking": False, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 36.1461, "lon": 137.2517,
     "tags": ["alpine", "mountain", "rural", "historic", "traditional",
             "festival", "sake_brewery", "edo_town", "crafts", "serene",
             "hiking", "wood_carving", "morning_market"]},

    {"region": "Shirakawa-go", "is_anchor": False, "tourist_count": 22000, "capacity": 20000,
     "avg_delay_minutes": 15, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 50, "promotion_score": 70, "cleanliness_score": 80,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 36.2578, "lon": 136.9061,
     "tags": ["rural", "historic", "snow", "mountain", "unesco",
             "traditional", "thatched_roof", "village", "photo_spot", "serene",
             "farming", "culture", "winter"]},

    {"region": "Koyasan", "is_anchor": False, "tourist_count": 12000, "capacity": 25000,
     "avg_delay_minutes": 5, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 60, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 34.2130, "lon": 135.5830,
     "tags": ["temple", "historic", "mountain", "rural", "unesco",
             "buddhist", "cemetery", "serene", "meditation", "cable_car",
             "traditional", "spiritual", "pilgrim", "shojin_ryori"]},

    {"region": "Nagasaki", "is_anchor": False, "tourist_count": 20000, "capacity": 45000,
     "avg_delay_minutes": 8, "cultural_score": 88, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 32.7503, "lon": 129.8779,
     "tags": ["historic", "memorial", "coastal", "food", "tram",
             "culture", "church", "port", "island", "museum",
             "solemn", "international", "night_view"]},

    {"region": "Tottori", "is_anchor": False, "tourist_count": 12000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 70, "experiential_score": 78,
     "infrastructure_score": 60, "promotion_score": 40, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 35.5011, "lon": 134.2351,
     "tags": ["rural", "nature", "sand_dunes", "coastal", "adventure",
             "camel_ride", "paragliding", "museum", "quiet", "scenic",
             "seafood", "off_beaten_path"]},

    {"region": "Akita", "is_anchor": False, "tourist_count": 9000, "capacity": 30000,
     "avg_delay_minutes": 3, "cultural_score": 65, "experiential_score": 72,
     "infrastructure_score": 55, "promotion_score": 35, "cleanliness_score": 82,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 39.7186, "lon": 140.1024,
     "tags": ["rural", "nature", "festival", "mountain", "snow",
             "onsen", "lake", "samurai", "rice_paddy", "quiet",
             "traditional", "rustic"]},

    {"region": "Shimane", "is_anchor": False, "tourist_count": 7000, "capacity": 25000,
     "avg_delay_minutes": 2, "cultural_score": 75, "experiential_score": 70,
     "infrastructure_score": 50, "promotion_score": 30, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 35.4723, "lon": 133.0505,
     "tags": ["historic", "shrine", "garden", "rural", "mythology",
             "traditional", "serene", "castle", "quiet", "culture",
             "off_beaten_path", "sunset"]},

    {"region": "Okayama", "is_anchor": False, "tourist_count": 15000, "capacity": 40000,
     "avg_delay_minutes": 5, "cultural_score": 82, "experiential_score": 75,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.6551, "lon": 133.9195,
     "tags": ["historic", "garden", "castle", "culture", "traditional",
             "crafts", "denim", "fruit", "cycling", "serene",
             "architecture", "museum"]},

    {"region": "Kagawa", "is_anchor": False, "tourist_count": 14000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 82,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.3401, "lon": 134.0434,
     "tags": ["food", "coastal", "shrine", "udon", "island",
             "art", "garden", "pilgrimage", "scenic", "cycling",
             "bridge", "traditional"]},

    {"region": "Naoshima", "is_anchor": False, "tourist_count": 11000, "capacity": 15000,
     "avg_delay_minutes": 12, "cultural_score": 88, "experiential_score": 90,
     "infrastructure_score": 55, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 34.4614, "lon": 133.9958,
     "tags": ["art", "coastal", "modern", "rural", "museum",
             "architecture", "island", "sculpture", "gallery", "photo_spot",
             "minimalist", "design", "cycling"]},

    {"region": "Matsuyama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 85, "experiential_score": 82,
     "infrastructure_score": 72, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.8392, "lon": 132.7657,
     "tags": ["hot_spring", "historic", "castle", "tram", "traditional",
             "literature", "garden", "relaxation", "food", "serene",
             "architecture", "culture"]},

    {"region": "Beppu", "is_anchor": False, "tourist_count": 25000, "capacity": 45000,
     "avg_delay_minutes": 10, "cultural_score": 75, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.2846, "lon": 131.4914,
     "tags": ["hot_spring", "nature", "coastal", "volcanic", "steam",
             "relaxation", "ryokan", "sand_bath", "scenic", "food",
             "hell_tour", "unique"]},

    {"region": "Kurokawa Onsen", "is_anchor": False, "tourist_count": 8000, "capacity": 12000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 45, "promotion_score": 40, "cleanliness_score": 92,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 33.1283, "lon": 131.0714,
     "tags": ["hot_spring", "rural", "mountain", "ryokan", "serene",
             "traditional", "nature", "relaxation", "lantern", "rustic",
             "intimate", "off_beaten_path"]},

    {"region": "Kagoshima", "is_anchor": False, "tourist_count": 16000, "capacity": 38000,
     "avg_delay_minutes": 5, "cultural_score": 78, "experiential_score": 85,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 83,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 31.5966, "lon": 130.5571,
     "tags": ["nature", "coastal", "mountain", "hot_spring", "volcanic",
             "scenic", "food", "garden", "historic", "tram",
             "island", "adventure", "subtropical"]},

    {"region": "Sendai", "is_anchor": False, "tourist_count": 30000, "capacity": 65000,
     "avg_delay_minutes": 8, "cultural_score": 82, "experiential_score": 80,
     "infrastructure_score": 85, "promotion_score": 60, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 38.2682, "lon": 140.8694,
     "tags": ["urban", "historic", "food", "festival", "castle",
             "shopping", "beef_tongue", "tanabata", "park", "modern",
             "day_trip", "lively"]},

    {"region": "Aomori", "is_anchor": False, "tourist_count": 10000, "capacity": 35000,
     "avg_delay_minutes": 3, "cultural_score": 75, "experiential_score": 80,
     "infrastructure_score": 60, "promotion_score": 45, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 40.8244, "lon": 140.7400,
     "tags": ["nature", "snow", "rural", "festival", "nebuta",
             "apple", "hiking", "mountain", "lake", "scenic",
             "autumn_leaves", "quiet", "rustic"]},

    {"region": "Nagano", "is_anchor": False, "tourist_count": 22000, "capacity": 50000,
     "avg_delay_minutes": 7, "cultural_score": 85, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 60, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.6513, "lon": 138.1810,
     "tags": ["mountain", "snow", "nature", "temple", "ski",
             "monkey_park", "hiking", "alpine", "traditional", "soba",
             "scenic", "onsen", "winter_sport"]},

    {"region": "Matsumoto", "is_anchor": False, "tourist_count": 15000, "capacity": 35000,
     "avg_delay_minutes": 5, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 55, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.2380, "lon": 137.9720,
     "tags": ["castle", "historic", "alpine", "traditional", "culture",
             "crafts", "soba", "museum", "architecture", "serene",
             "mountain", "wasabi", "art"]},

    {"region": "Toyama", "is_anchor": False, "tourist_count": 13000, "capacity": 40000,
     "avg_delay_minutes": 4, "cultural_score": 75, "experiential_score": 85,
     "infrastructure_score": 68, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.6953, "lon": 137.2114,
     "tags": ["alpine", "nature", "coastal", "snow", "scenic",
             "glass_art", "sushi", "firefly_squid", "mountain", "dam",
             "hiking", "modern", "quiet"]},
]

df = pd.DataFrame(data)

# Quick validation
tag_counts = df["tags"].apply(len)
print(f"Locations: {len(df)}  |  Anchors: {df['is_anchor'].sum()}  |  Orbits: {(~df['is_anchor']).sum()}")
print(f"Tags per location - min: {tag_counts.min()}, max: {tag_counts.max()}, mean: {tag_counts.mean():.1f}")
print(f"Coordinates present: {df['lat'].notna().all() and df['lon'].notna().all()}")
df[["region", "is_anchor", "lat", "lon", "cultural_score"]].head(5)


Locations: 30  |  Anchors: 10  |  Orbits: 20
Tags per location - min: 12, max: 15, mean: 12.8
Coordinates present: True


,region,is_anchor,lat,lon,cultural_score
0,Tokyo,True,35.6895,139.6917,85
1,Kyoto,True,35.0116,135.7681,95
2,Osaka,True,34.6937,135.5023,80
3,Hakone,True,35.2326,139.1070,85
4,Nara,True,34.6851,135.8048,95


## The Scoring Formula

This markdown cell documents the formal objective function that the recommendation engine optimises. The composite score for each candidate destination is a weighted sum of three components: TF-IDF semantic similarity (how well the tags match), KNN profile similarity (how similar the quality features are), and an attractiveness score. Two penalties are subtracted: a congestion penalty that is deliberately asymmetric (it *rewards* undertouristed destinations and *penalises* overcrowded ones), and a logistical penalty that accounts for travel time and group feasibility. The asymmetric congestion penalty is the key redistribution mechanism. If you are curious about the exact piecewise structure, it is defined right here.


# Formal Objective Function

## Decision Variable

For each candidate destination $i$ in the survivor set, compute a composite score $F_i$:

$$F_i = (\alpha \cdot S_i + \beta \cdot P_i + \gamma \cdot A_i) - \Psi(L_i) - \delta \cdot C_i$$

## Component Definitions

| Symbol | Definition | Source |
|--------|-----------|--------|
| $S_i$  | Semantic similarity score (TF-IDF cosine, renormalised within survivor set to [0, 100]) | `_compute_tfidf_scores()` |
| $P_i$  | Profile similarity score (KNN cosine distance on quality features, scaled to [0, 100]) | `_compute_knn_profile_scores()` |
| $A_i$  | Attractiveness score (weighted combination of cultural, experiential, cleanliness, infrastructure scores — weights vary by `tourist_type`) | Inline in `generate_recommendations()` |
| $L_i$  | Congestion load index (piecewise function of `tourist_count / capacity` and `avg_delay_minutes`) | `compute_congestion_index()` |
| $C_i$  | Capacity-adjusted spatial cost: $C_i = \frac{\text{commute\_minutes}_i}{1 - \hat{r}_i}$ where $\hat{r}_i = \min(r_i, 0.95)$ is the clamped load ratio. The 0.95 clamp prevents division by zero and sign inversion when $r_i \geq 1.0$. | `generate_recommendations()` |

## Weight Parameters

| Weight | Default | Constraint |
|--------|---------|------------|
| $\alpha$ | 0.50 | $\alpha + \beta + \gamma = 1$ |
| $\beta$  | 0.30 | |
| $\gamma$  | 0.20 | |
| $\delta$  | 0.05 | Tunable; caps $\delta \cdot C_i$ at 20 points max |

## Asymmetric Congestion Penalty $\Psi(L_i)$

$$\Psi(L_i) = \begin{cases}
-15.0       & \text{if } L_i < 20 \quad \textit{(Severely undertouristed — active bonus)} \\
-8.0        & \text{if } L_i < 40 \quad \textit{(Undertouristed — small bonus)} \\
L_i \times 0.20  & \text{if } L_i < 75 \quad \textit{(Balanced — gentle deterrent)} \\
L_i \times 0.50  & \text{if } L_i < 100 \quad \textit{(Approaching overtourism)} \\
L_i \times 0.75  & \text{if } L_i \geq 100 \quad \textit{(Severely overtouristed — maximum deterrent)}
\end{cases}$$

**Design intent:** $\Psi$ is intentionally asymmetric. Negative values **actively lift** undertouristed destinations in the ranking; positive values **deter** overtouristed ones. This asymmetry is the core redistribution mechanism, not a calibration choice.

## Post-Formula Logistical Penalty

Logistical penalties (`dynamic_penalty` + `distance_penalty`) are applied **post-formula** and **capped at 40% of base score** to prevent infrastructure friction from overriding the redistribution signal.

$$\text{logistical\_penalty} = \min(\text{dynamic\_penalty} + \text{distance\_penalty},\; 0.40 \times \text{base\_score})$$

**Congestion penalty $\Psi(L_i)$ is never capped** — it is the core redistribution signal.

$$\text{final\_match\_score} = \text{base\_score} - \Psi(L_i) - \text{logistical\_penalty}$$

## Assumptions

1. **Linearity and additive separability**: The objective assumes component scores contribute additively and can be meaningfully summed on a common scale.
2. **Heuristic aggregation**: This is a heuristic multi-objective aggregation, not a globally optimal solution. No Pareto frontier is computed.
3. $\alpha, \beta, \gamma, \delta$ are **tunable hyperparameters** — the defaults reflect domain judgement, not empirical calibration.
4. The mathematical definition above **precisely mirrors** the implemented piecewise structure in `congestion_penalty()`. Any divergence between this formula and the code is a documentation error, not a modelling choice.

## Assumptions and Limitations

A brief but important disclaimer. The dataset is a simulation sandbox, not empirical data. The scores, counts, and capacity figures are designed to span realistic ranges but are not sourced from any real survey or API. Results should be interpreted as algorithmic demos, not policy prescriptions. The cell also lists what production deployment would require: real mobility data, live capacity feeds, validated preference surveys, and seasonal factors.


# Simulation Environment Assumptions

This notebook operates on a **synthetic dataset** constructed for algorithmic evaluation purposes. The following assumptions govern the experimental design:

1. **Synthetic sandbox**: The dataset functions as a controlled experimental sandbox for testing redistribution logic under variable congestion dynamics. It is **not** an empirical representation of actual Japan tourism flows.

2. **Illustrative scores**: Score values (`cultural_score`, `infrastructure_score`, `experiential_score`, `cleanliness_score`) are illustrative parameters designed to span a realistic range. They are **not** validated measurements from field surveys or official statistics.

3. **Deliberate construction**: The synthetic structure is a methodological choice that allows controlled manipulation of congestion variables (tourist_count, capacity, avg_delay_minutes) without confounding real-world data noise. Anchor/orbit asymmetry is designed to test the redistribution signal under known conditions.

4. **Real-world deployment requirements**: Production deployment would require integration with:
   - Mobility datasets (e.g., mobile location data, transit card records)
   - Live capacity feeds from destination management organisations
   - Validated tourist preference surveys (conjoint analysis or revealed preference)
   - Seasonal adjustment factors not present in this static snapshot

5. **No empirical claims**: Results produced by this model should be interpreted as algorithmic demonstrations, not tourism policy recommendations. All "tourist_count" values are simulation parameters, not observed visitor counts.

## Measuring Crowding and Mapping Travel Routes

Two core utilities live here. First, the congestion index: a piecewise function that takes a destination's load ratio (tourist_count / capacity) and average delay, and outputs a single congestion score. Below 0.3 load ratio it rises gently; above 1.0 it escalates aggressively with no ceiling. Second, the A* spatial graph: a NetworkX graph modelling Japan's transit network with realistic edge weights in minutes. The A* search uses Haversine great-circle distance as an admissible heuristic to find shortest travel times between any two destinations. This gives us actual route-based commute times instead of crude straight-line distances.


In [16]:
# ── Phase 1 Congestion Helpers ────────────────────────────────────────────────
def piecewise_load_score(ratio: float) -> float:
    """Piecewise congestion score with no ceiling above ratio >= 1.2."""
    if ratio < 0.3:
        return ratio * (20 / 0.3)
    elif ratio < 0.7:
        return 20 + (ratio - 0.3) * (30 / 0.4)
    elif ratio < 0.9:
        return 50 + (ratio - 0.7) * (25 / 0.2)
    elif ratio < 1.0:
        return 75 + (ratio - 0.9) * (15 / 0.1)
    elif ratio < 1.2:
        return 90 + (ratio - 1.0) * (10 / 0.2)
    else:
        return 100 + ((ratio - 1.2) * 20)


def compute_congestion_index(load_score: float, delay_score: float) -> float:
    """Dominance Rule: delay amplifies when load_score >= 100."""
    if load_score >= 100:
        return load_score + (delay_score * 0.20)
    return (load_score * 0.70) + (delay_score * 0.30)


def congestion_flag(index: float) -> str:
    if index < 20:   return "Severely undertouristed"
    elif index < 40: return "Undertouristed"
    elif index < 75: return "Balanced"
    elif index < 100: return "Approaching overtourism"
    else:             return "Severely overtouristed"


def compute_dynamic_penalty(row: pd.Series, group_size: int) -> int:
    """Group-size-gated feasibility penalty."""
    if group_size >= 15:
        return (
            (0 if row["has_coach_parking"]  else 20) +
            (0 if row["group_dining_nearby"] else 15) +
            (10 if row["requires_long_walk"] else 0)
        )
    else:
        return (5 if row["requires_long_walk"] else 0)


# ── Upgrade 1: Asymmetric Piecewise Congestion Penalty Ψ(L_i) ──────────────────
# This function mirrors the formal objective function definition exactly.
# Negative returns are active bonuses for undertouristed destinations.
# Positive returns are deterrents for overtouristed destinations.
def congestion_penalty(index: float) -> float:
    """Asymmetric congestion penalty Ψ(L_i) — the core redistribution mechanism.
    
    Negative values actively LIFT undertouristed destinations.
    Positive values DETER overtouristed destinations.
    This penalty is NEVER capped — it is the redistribution signal."""
    if index < 20:
        # Severely undertouristed — active bonus lifts these destinations
        return -15.0
    elif index < 40:
        # Undertouristed — small bonus encourages redistribution here
        return -8.0
    elif index < 75:
        # Balanced — gentle deterrent proportional to congestion
        return index * 0.20
    elif index < 100:
        # Approaching overtourism — stronger deterrent
        return index * 0.50
    else:
        # Severely overtouristed — maximum deterrent
        return index * 0.75


# ── Phase 3: A* Spatial Graph ─────────────────────────────────────────────────
def build_japan_astar_graph(df: pd.DataFrame) -> nx.Graph:
    """
    Build a fully-connected weighted graph of Japan's transit network.
    Edge weights represent realistic travel times in minutes.
    """
    G = nx.Graph()

    # Add nodes with lat/lon attributes
    for _, row in df.iterrows():
        G.add_node(row["region"], lat=row["lat"], lon=row["lon"])

    # ── Tokaido-Sanyo Shinkansen Spine ────────────────────────────────────
    G.add_edge("Tokyo",      "Hakone",     weight=40)
    G.add_edge("Hakone",     "Kyoto",      weight=120)
    G.add_edge("Kyoto",      "Osaka",      weight=15)
    G.add_edge("Osaka",      "Nara",       weight=40)
    G.add_edge("Osaka",      "Hiroshima",  weight=90)
    G.add_edge("Hiroshima",  "Fukuoka",    weight=65)

    # ── Kanto Branches ────────────────────────────────────────────────────
    G.add_edge("Tokyo",      "Kamakura",   weight=55)
    G.add_edge("Tokyo",      "Nikko",      weight=110)

    # ── Tohoku Shinkansen ─────────────────────────────────────────────────
    G.add_edge("Tokyo",      "Sendai",     weight=90)
    G.add_edge("Sendai",     "Aomori",     weight=100)
    G.add_edge("Sendai",     "Akita",      weight=110)

    # ── Hokkaido ──────────────────────────────────────────────────────────
    G.add_edge("Aomori",     "Sapporo",    weight=300)

    # ── Hokuriku / Central Japan ──────────────────────────────────────────
    G.add_edge("Kyoto",      "Kanazawa",   weight=130)
    G.add_edge("Kanazawa",   "Toyama",     weight=25)
    G.add_edge("Kanazawa",   "Shirakawa-go", weight=60)
    G.add_edge("Shirakawa-go", "Takayama", weight=50)
    G.add_edge("Tokyo",      "Nagano",     weight=90)
    G.add_edge("Nagano",     "Matsumoto",  weight=50)
    G.add_edge("Matsumoto",  "Takayama",   weight=120)

    # ── Kansai Branches ───────────────────────────────────────────────────
    G.add_edge("Osaka",      "Koyasan",    weight=90)

    # ── San'yo / San'in ───────────────────────────────────────────────────
    G.add_edge("Hiroshima",  "Okayama",    weight=40)
    G.add_edge("Okayama",    "Tottori",    weight=120)
    G.add_edge("Tottori",    "Shimane",    weight=120)

    # ── Shikoku ───────────────────────────────────────────────────────────
    G.add_edge("Okayama",    "Kagawa",     weight=55)
    G.add_edge("Kagawa",     "Naoshima",   weight=30)
    G.add_edge("Kagawa",     "Matsuyama",  weight=150)

    # ── Kyushu ────────────────────────────────────────────────────────────
    G.add_edge("Fukuoka",    "Nagasaki",   weight=110)
    G.add_edge("Fukuoka",    "Beppu",      weight=120)
    G.add_edge("Fukuoka",    "Kagoshima",  weight=75)
    G.add_edge("Beppu",      "Kurokawa Onsen", weight=90)

    return G


def haversine_heuristic(u, v, d) -> float:
    """
    A* admissible heuristic: Haversine great-circle distance
    converted to an optimistic time estimate at Shinkansen speed (300 km/h).
    """
    lat1, lon1 = d[u]["lat"], d[u]["lon"]
    lat2, lon2 = d[v]["lat"], d[v]["lon"]

    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance_km = R * c

    heuristic_time = (distance_km / 300.0) * 60.0
    return heuristic_time


def get_astar_commute_minutes(graph: nx.Graph, anchor: str, orbit: str) -> float:
    """Return A* shortest-path travel time (minutes) between anchor and orbit."""
    try:
        return nx.astar_path_length(
            graph,
            source=anchor,
            target=orbit,
            heuristic=lambda u, v: haversine_heuristic(u, v, graph.nodes),
            weight="weight",
        )
    except nx.NetworkXNoPath:
        return float("inf")


print("Phase 1 congestion helpers (Ψ upgraded) + Phase 3 A* spatial graph loaded.")


Phase 1 congestion helpers (Ψ upgraded) + Phase 3 A* spatial graph loaded.


## Ethical Guardrails

A pre-filter that gates every candidate before it reaches the scoring engine. Five constraints, checked in order: (1) emergency blacklist, rejecting regions hit by typhoons, protests, or other high-severity events; (2) capacity ceiling at 80%, preventing secondary congestion; (3) infrastructure floor at 55, ensuring the destination can physically handle groups; (4) cultural floor at 60, avoiding poor visitor experiences; (5) viability floor at 5% load, filtering out operationally dormant destinations. The blacklist check was added as part of the Phase 4 temporal layer and runs before all other constraints.


In [17]:
# ── Upgrade 5: Ethical Constraint Layer ───────────────────────────────────────

# Constraint thresholds — each with rationale
CAPACITY_CEILING = 0.80       # Do not redirect if projected load > 80% capacity
INFRASTRUCTURE_FLOOR = 55     # Minimum infrastructure score for safe absorption
CULTURAL_FLOOR = 60           # Minimum cultural score to avoid degrading authenticity
VIABILITY_FLOOR_RATIO = 0.05  # Minimum operational load (5% of capacity)


def check_ethical_constraints(row, df_sim, expected_redirected=0,
                              blacklisted_regions=None):
    """
    Pre-filter destinations before scoring. Returns True if destination passes
    ALL constraints, False if any constraint is violated.
    
    Parameters
    ----------
    row : pd.Series — candidate destination row
    df_sim : pd.DataFrame — current simulation state (for context)
    expected_redirected : int — tourists expected to be redirected this round
    blacklisted_regions : list or None — regions blacklisted by emergency events
    """
    # Constraint 0 — Emergency Blacklist (Phase 4):
    # Blacklisted regions are unconditionally excluded regardless of scores.
    # This check MUST run before all other constraints.
    if blacklisted_regions and row["region"] in blacklisted_regions:
        return False
    
    # Constraint 1 — Capacity Ceiling:
    # Prevents creating secondary congestion at alternatives.
    # The 80% threshold preserves buffer for organic tourist arrivals.
    projected_load = (row["tourist_count"] + expected_redirected) / row["capacity"]
    if projected_load > CAPACITY_CEILING:
        return False
    
    # Constraint 2 — Infrastructure Floor:
    # Below 55, destinations lack baseline facilities (transport, sanitation,
    # emergency services) to safely absorb redirected volumes.
    if row["infrastructure_score"] < INFRASTRUCTURE_FLOOR:
        return False
    
    # Constraint 3 — Cultural Sensitivity Floor:
    # Redirecting large volumes to culturally fragile destinations risks
    # degrading the authenticity that makes them worth preserving.
    if row["cultural_score"] < CULTURAL_FLOOR:
        return False
    
    # Constraint 4 — Minimum Viability Floor:
    # Destinations far below viable load may lack activated services.
    # Redirecting tourists to operationally dormant destinations produces
    # poor visitor outcomes and undermines model credibility.
    if row["tourist_count"] < VIABILITY_FLOOR_RATIO * row["capacity"]:
        return False
    
    return True


print("Ethical constraint layer loaded (5 constraints, pre-filter).") 


Ethical constraint layer loaded (5 constraints, pre-filter).


## Seasons, Festivals, and Emergencies (Phase 4)

This cell adds time-awareness to the system. It defines seasonal multipliers for five regions (e.g., Kyoto peaks at 2.8x in April for cherry blossoms, Sapporo at 2.5x in February for the snow festival), and a static event catalog with five entries covering festivals and emergencies.

The main function, `apply_temporal_adjustments`, does three things. First, it scales tourist_count by the seasonal multiplier for the travel month. Second, it overlays active events: emergencies add congestion and reduce attractiveness (with high-severity ones blacklisting the region entirely), while festivals use a preference-gating mechanism. For anchor festivals, if the tourist's tag overlap with the festival's tags meets a 40% threshold, the system assumes they are there *for* the festival and boosts attractiveness without adding congestion. If the overlap is below 40%, the tourist does not care about the festival, so only the congestion delta is applied, increasing redirect pressure. Orbit festivals always apply both signals. Third, a booking-horizon decay function increases urgency for last-minute bookings.

Event adjustments are stored in additive delta columns (`event_congestion_delta`, `event_attractiveness_delta`) rather than mutating base scores, which keeps multi-round simulations stable.


In [18]:
# ── Phase 4: Temporal & Event Layer ──────────────────────────────────────────

# ── 4A. CONSTANTS ────────────────────────────────────────────────────────────

# SEASONAL_MULTIPLIERS: region → {month_int: multiplier}
# Reflects peak/off-peak tourism patterns across Japan.
# Destinations not listed default to 1.0 (no seasonal adjustment).
SEASONAL_MULTIPLIERS = {
    "Kyoto":    {3: 2.0, 4: 2.8, 5: 1.6, 10: 1.8, 11: 2.2, 12: 0.7},
    "Sapporo":  {1: 2.0, 2: 2.5, 7: 1.5, 8: 1.6},
    "Nara":     {3: 1.8, 4: 2.5, 11: 1.6},
    "Okinawa":  {7: 2.2, 8: 2.5, 1: 0.6},
    "Kanazawa": {4: 1.8, 10: 1.5},
}

# EVENT_SOURCE_MODE: "static" uses STATIC_EVENT_CATALOG below.
# Switch to "live_feed" when Japan Tourism Agency API / NHK alert feed available.
# Application logic never changes — only the data source.
EVENT_SOURCE_MODE = "static"

# FESTIVAL_STAY_THRESHOLD: minimum tag overlap ratio for a tourist to
# stay at the anchor during a festival. Below this, redirect pressure increases.
FESTIVAL_STAY_THRESHOLD = 0.40

# STATIC_EVENT_CATALOG: list of event dicts following the canonical schema.
# Each event uses month integers (not ISO dates) for direct travel_month comparison.
#   attractiveness_delta: positive = more attractive, negative = less attractive.
#                         Emergency entries MUST use negative values.
STATIC_EVENT_CATALOG = [
    # ── Kyoto: Gion Matsuri (July, high severity) ───────────────────────────
    {
        "event_id": "gion_matsuri_2025",
        "region": "Kyoto",
        "event_type": "festival",
        "severity": "high",
        "start_month": 7,
        "end_month": 7,
        "congestion_delta": 25,
        "attractiveness_delta": 20,
        "redirect_mode": "preference_gated",
        "affected_tags": ["festival", "traditional", "culture", "parade", "summer"],
    },
    # ── National: Golden Week (May, multiple regions) ───────────────────────
    {
        "event_id": "golden_week_2025",
        "region": "Tokyo",
        "event_type": "festival",
        "severity": "high",
        "start_month": 5,
        "end_month": 5,
        "congestion_delta": 30,
        "attractiveness_delta": 15,
        "redirect_mode": "preference_gated",
        "affected_tags": ["urban", "shopping", "food", "modern", "lively"],
    },
    # ── Emergency: protest (medium severity) ────────────────────────────────
    {
        "event_id": "osaka_protest_2025",
        "region": "Osaka",
        "event_type": "emergency",
        "severity": "medium",
        "start_month": 6,
        "end_month": 6,
        "congestion_delta": 15,
        "attractiveness_delta": -10,
        "redirect_mode": "unconditional",
        "affected_tags": [],
    },
    # ── Emergency: typhoon warning (high severity, coastal) ─────────────────
    {
        "event_id": "typhoon_warning_2025",
        "region": "Kagoshima",
        "event_type": "emergency",
        "severity": "high",
        "start_month": 8,
        "end_month": 9,
        "congestion_delta": 20,
        "attractiveness_delta": -25,
        "redirect_mode": "unconditional",
        "affected_tags": [],
    },
    # ── Orbit festival: Kanazawa Hyakumangoku (June, low severity) ──────────
    # Makes an undertouristed destination more attractive as redirect target.
    {
        "event_id": "kanazawa_hyakumangoku_2025",
        "region": "Kanazawa",
        "event_type": "festival",
        "severity": "low",
        "start_month": 6,
        "end_month": 6,
        "congestion_delta": 10,
        "attractiveness_delta": 15,
        "redirect_mode": "preference_gated",
        "affected_tags": ["festival", "traditional", "parade", "samurai", "culture"],
    },
]


# ── 4B. TEMPORAL ADJUSTMENT FUNCTION ──────────────────────────────────────────

def apply_temporal_adjustments(df_sim, travel_month, tourist_tags, days_ahead,
                               anchor_region):
    """
    Apply seasonal, event, and booking-horizon adjustments to a working copy of df.
    
    Parameters
    ----------
    df_sim        : pd.DataFrame — working copy (already copied before this call)
    travel_month  : int (1-12)
    tourist_tags  : list of str — tourist preference tags (empty list if None)
    days_ahead    : int — days between booking and travel date
    anchor_region : str — the anchor region name, for anchor vs orbit festival logic
    
    Returns
    -------
    (adjusted_df, blacklisted_regions) : tuple
    """
    # ── Step 1: Seasonal adjustment ──────────────────────────────────────────
    for idx, row in df_sim.iterrows():
        multiplier = SEASONAL_MULTIPLIERS.get(row["region"], {}).get(travel_month, 1.0)
        df_sim.at[idx, "tourist_count"] = row["tourist_count"] * multiplier

    # ── Step 2: Event overlay ────────────────────────────────────────────────
    # Initialize event columns as additive deltas (never mutate base scores)
    if "event_congestion_delta" not in df_sim.columns:
        df_sim["event_congestion_delta"] = 0
    if "event_attractiveness_delta" not in df_sim.columns:
        df_sim["event_attractiveness_delta"] = 0

    # FIX 1: Use load_events() — the switchable data source scaffold.
    # Changing EVENT_SOURCE_MODE to "live_feed" now correctly routes here.
    active_events = load_events(mode=EVENT_SOURCE_MODE, travel_month=travel_month)

    blacklisted_regions = []

    for event in active_events:
        region_mask = df_sim["region"] == event["region"]
        if not region_mask.any():
            continue

        if event["event_type"] == "emergency":
            # Add congestion_delta to event column (not tourist_count)
            df_sim.loc[region_mask, "event_congestion_delta"] += event["congestion_delta"]
            # FIX 3: No abs() — catalog convention enforces negative values
            # for emergencies, so += with a negative value correctly reduces.
            df_sim.loc[region_mask, "event_attractiveness_delta"] += event[
                "attractiveness_delta"
            ]
            if event["severity"] in ("high", "critical"):
                blacklisted_regions.append(event["region"])

        elif event["event_type"] == "festival":
            is_anchor_festival = (event["region"] == anchor_region)

            if is_anchor_festival:
                # Compute match ratio between tourist tags and festival tags
                if len(event["affected_tags"]) > 0:
                    match_ratio = (
                        len(set(tourist_tags) & set(event["affected_tags"]))
                        / len(event["affected_tags"])
                    )
                else:
                    match_ratio = 0.0

                if match_ratio >= FESTIVAL_STAY_THRESHOLD:
                    # Tourist aligns — do NOT redirect, boost attractiveness
                    df_sim.loc[region_mask, "event_attractiveness_delta"] += event[
                        "attractiveness_delta"
                    ]
                    # Do NOT apply congestion_delta
                else:
                    # Tourist does not align — heighten redirect pressure
                    df_sim.loc[region_mask, "event_congestion_delta"] += event[
                        "congestion_delta"
                    ]
            else:
                # Orbit festival — both signals present
                df_sim.loc[region_mask, "event_attractiveness_delta"] += event[
                    "attractiveness_delta"
                ]
                df_sim.loc[region_mask, "event_congestion_delta"] += event[
                    "congestion_delta"
                ]

    # ── Step 3: Advance booking horizon ──────────────────────────────────────
    # At days_ahead=0  → 1.50 (same-day, maximum urgency)
    # At days_ahead=30 → ~1.18
    # At days_ahead=90 → ~1.02 (well in advance, minimal effect)
    booking_trend = 1.0 + (0.5 * math.exp(-days_ahead / 30.0))
    df_sim["tourist_count"] = df_sim["tourist_count"] * booking_trend

    return (df_sim, blacklisted_regions)


# ── 4C. LIVE FEED SCAFFOLD ───────────────────────────────────────────────────

def load_events(mode=EVENT_SOURCE_MODE, travel_month=None):
    """Load active events for the given travel month.
    
    mode="static" returns from STATIC_EVENT_CATALOG.
    mode="live_feed" is reserved for future API integration.
    """
    if mode == "static":
        return [
            e for e in STATIC_EVENT_CATALOG
            if travel_month is None
            or e["start_month"] <= travel_month <= e["end_month"]
        ]
    elif mode == "live_feed":
        # Future: Japan Tourism Agency API, NHK alert feed,
        # prefecture emergency systems.
        raise NotImplementedError("Live feed not yet connected.")


print("Phase 4: Temporal & Event Layer loaded.")


Phase 4: Temporal & Event Layer loaded.


## The Recommendation Engine

The core of the notebook. Given an anchor, group size, tourist type, and optionally a travel month with tourist tags, it outputs a ranked DataFrame of alternative destinations.

The pipeline: (1) Build the A* transit graph; (2) if travel_month is set, apply temporal adjustments on a fresh df.copy(); (3) compute TF-IDF cosine similarity between the anchor's tags and every other destination's tags, then min-max normalise within the survivor set; (4) run KNN on numerical quality features for profile-based re-ranking; (5) score each survivor using the formal objective function (weighted sum of TF-IDF, KNN, and attractiveness minus congestion penalty and logistical penalty). The `event_congestion_delta` column, if present, feeds additively into the congestion index. The `event_attractiveness_delta` is added to experiential_score before attractiveness weighting, clamped to [0, 100]. Logistical penalties (group feasibility + distance) are capped at 40% of base score so they never completely override a strong match.


In [19]:
# ── Two-Stage NLP + A* Spatial Graph Recommendation Engine ─────────────────────
# Upgrade 7: Spatial cost now feeds into final score (capacity-adjusted).
# Upgrade 5: Ethical constraints pre-filter destinations before scoring.
# Phase 4: Temporal & Event adjustments applied before scoring.

# Spatial cost weight — commute time penalty adjusted by destination availability
SPATIAL_COST_WEIGHT = 0.05  # δ: tunable, caps distance_penalty at 20 max


def _compute_tfidf_scores(df: pd.DataFrame, anchor_name: str) -> dict:
    """Stage 1: TF-IDF cosine similarity (anchor vs all). Returns {region: score}."""
    tag_strings = df["tags"].apply(lambda t: " ".join(t))
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(tag_strings)
    anchor_idx = df.index[df["region"] == anchor_name][0]
    cos_sim = cosine_similarity(tfidf_matrix[anchor_idx], tfidf_matrix).flatten()
    return dict(zip(df["region"], np.round(cos_sim * 100, 2)))


def _compute_knn_profile_scores(df: pd.DataFrame, anchor_name: str,
                                 survivor_regions: list) -> dict:
    """Stage 2: KNN cosine-distance similarity on numerical features.
    Only scores survivors from Stage 1. Returns {region: score}."""
    num_cols = ["cultural_score", "experiential_score",
               "cleanliness_score", "infrastructure_score"]
    scaler = MinMaxScaler()
    num_matrix = scaler.fit_transform(df[num_cols].values)

    knn = NearestNeighbors(n_neighbors=len(df), metric="cosine")
    knn.fit(num_matrix)

    anchor_idx = df.index[df["region"] == anchor_name][0]
    distances, indices = knn.kneighbors(num_matrix[anchor_idx].reshape(1, -1))

    sim_map = {}
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        region = df.iloc[idx]["region"]
        if region in survivor_regions:
            sim_map[region] = round((1 - dist) * 100, 2)
    return sim_map


# Column schema for the output DataFrame (updated with distance_penalty)
_OUTPUT_COLS = [
    "region", "tfidf_score", "knn_profile_score",
    "congestion_index", "congestion_flag",
    "dynamic_penalty", "distance_penalty", "commute_minutes",
    "total_penalty", "final_match_score",
]


def generate_recommendations(
    df: pd.DataFrame,
    anchor_region: str,
    group_size: int,
    tourist_type: str,
    travel_month=None,
    tourist_tags=None,
    days_ahead=30,
) -> pd.DataFrame:
    """
    Two-Stage NLP Recommendation Engine with:
    - Phase 4 temporal & event adjustments
    - Ethical constraint pre-filtering (Upgrade 5) with emergency blacklist
    - Capacity-adjusted spatial cost (Upgrade 7)
    - Asymmetric congestion penalty Ψ (Upgrade 1)
    - Logistical penalty cap at 40% of base score
    
    Parameters travel_month, tourist_tags, days_ahead are optional.
    Defaults ensure backward compatibility with all existing callers.
    """
    tourist_type = tourist_type.lower().strip()
    if tourist_type not in ("international", "native"):
        raise ValueError("tourist_type must be 'international' or 'native'")

    anchor_row = df.loc[df["region"] == anchor_region]
    if anchor_row.empty:
        raise ValueError(f"Anchor region '{anchor_region}' not found.")

    # ── Build A* transit graph ─────────────────────────────────────────────
    transit_graph = build_japan_astar_graph(df)

    # ── Phase 4: Temporal & Event Adjustments ─────────────────────────────
    # df_working is ALWAYS a fresh copy — never operate on the original df.
    # When travel_month is None, temporal layer is inactive but df_working
    # still isolates this function from external mutation.
    blacklisted_regions = []
    df_working = df.copy()
    if travel_month is not None:
        t_tags = tourist_tags if tourist_tags is not None else []
        df_working, blacklisted_regions = apply_temporal_adjustments(
            df_working, travel_month, t_tags, days_ahead, anchor_region
        )

    # ── Stage 1: TF-IDF cosine similarity ─────────────────────────────────
    tfidf_scores = _compute_tfidf_scores(df_working, anchor_region)

    orbits = df_working[df_working["region"] != anchor_region].copy()
    orbits["tfidf_score"] = orbits["region"].map(tfidf_scores)
    survivors = orbits[orbits["tfidf_score"] > 0.0].copy()

    if survivors.empty:
        return pd.DataFrame(columns=_OUTPUT_COLS)

    # ── Upgrade 5: Ethical constraint pre-filtering ────────────────────────
    ethical_mask = survivors.apply(
        lambda row: check_ethical_constraints(
            row, df_working, blacklisted_regions=blacklisted_regions
        ),
        axis=1,
    )
    filtered_out = survivors[~ethical_mask]
    if not filtered_out.empty:
        for _, frow in filtered_out.iterrows():
            reasons = []
            # Check blacklist first
            if frow["region"] in blacklisted_regions:
                reasons.append("emergency blacklist (severity: high/critical)")
            else:
                lr = frow["tourist_count"] / frow["capacity"]
                if lr > CAPACITY_CEILING:
                    reasons.append(f"capacity ceiling ({lr:.0%} > {CAPACITY_CEILING:.0%})")
                if frow["infrastructure_score"] < INFRASTRUCTURE_FLOOR:
                    reasons.append(f"infrastructure floor ({frow['infrastructure_score']} < {INFRASTRUCTURE_FLOOR})")
                if frow["cultural_score"] < CULTURAL_FLOOR:
                    reasons.append(f"cultural floor ({frow['cultural_score']} < {CULTURAL_FLOOR})")
                if frow["tourist_count"] < VIABILITY_FLOOR_RATIO * frow["capacity"]:
                    reasons.append(f"viability floor ({frow['tourist_count']} < {VIABILITY_FLOOR_RATIO * frow['capacity']:.0f})")
            print(f"  ⊘ FILTERED: {frow['region']} — failed: {', '.join(reasons)}")
    
    survivors = survivors[ethical_mask].copy()
    
    if survivors.empty:
        print("  ⚠ WARNING: ALL alternatives failed ethical constraints. Returning empty.")
        return pd.DataFrame(columns=_OUTPUT_COLS)

    # ── FIX 1: Min-max renormalise TF-IDF within survivor set ─────────────
    tmin = survivors["tfidf_score"].min()
    tmax = survivors["tfidf_score"].max()
    if tmax > tmin:
        survivors["tfidf_score"] = (
            (survivors["tfidf_score"] - tmin) / (tmax - tmin) * 100
        ).round(2)
    else:
        survivors["tfidf_score"] = 75.0

    # ── Stage 2: KNN profile re-ranking ───────────────────────────────────
    survivor_list = survivors["region"].tolist()
    knn_scores = _compute_knn_profile_scores(df_working, anchor_region, survivor_list)
    survivors["knn_profile_score"] = survivors["region"].map(knn_scores)

    # ── Score each survivor ───────────────────────────────────────────────
    results = []
    for _, row in survivors.iterrows():

        # ── A* commute time ─────────────────────────────────────────────────
        commute = get_astar_commute_minutes(
            transit_graph, anchor_region, row["region"]
        )

        # ── Attractiveness weights ────────────────────────────────────────
        # Phase 4: event_attractiveness_delta is added and clamped to [0, 100]
        exp_score = row["experiential_score"] + row.get("event_attractiveness_delta", 0)
        exp_score = max(0, min(exp_score, 100))

        if tourist_type == "international":
            attractiveness = (
                row["cultural_score"]      * 0.40 +
                row["cleanliness_score"]   * 0.30 +
                exp_score                  * 0.20 +
                row["infrastructure_score"]* 0.10
            )
        else:
            attractiveness = (
                exp_score                  * 0.40 +
                row["infrastructure_score"]* 0.30 +
                row["cleanliness_score"]   * 0.20 +
                row["cultural_score"]      * 0.10
            )

        # ── Base score: F_i = α·S + β·P + γ·A ────────────────────────────
        base_score = (
            row["tfidf_score"]        * 0.50 +
            row["knn_profile_score"]  * 0.30 +
            attractiveness            * 0.20
        )

        # ── Congestion index ──────────────────────────────────────────────
        load_ratio  = row["tourist_count"] / row["capacity"]
        load_score  = piecewise_load_score(load_ratio)
        delay_score = min((row["avg_delay_minutes"] / 45) * 100, 100)
        c_index     = compute_congestion_index(load_score, delay_score)
        # Phase 4: incorporate event congestion delta (additive)
        c_index += row.get("event_congestion_delta", 0)
        c_index = max(c_index, 0)  # floor at zero
        c_flag      = congestion_flag(c_index)

        # ── Congestion penalty Ψ(L_i) — NEVER capped ─────────────────────
        c_penalty = congestion_penalty(c_index)

        # ── Feasibility penalty ───────────────────────────────────────────
        d_penalty = compute_dynamic_penalty(row, group_size)

        # ── Upgrade 7: Capacity-adjusted distance penalty ─────────────────
        # Guard: clamp load_ratio to 0.95 to avoid zero/negative denominator
        safe_load_ratio = min(load_ratio, 0.95)
        # Spatial utility: closer + emptier = better
        dist_penalty = SPATIAL_COST_WEIGHT * commute / (1 - safe_load_ratio)
        # Hard cap at 20 points maximum
        dist_penalty = round(min(dist_penalty, 20), 2)

        # ── Logistical penalty = dynamic + distance, capped at 40% base ──
        logistical_raw = round(d_penalty + dist_penalty, 2)
        logistical_capped = round(min(logistical_raw, 0.40 * base_score), 2)

        # ── Final score: base - Ψ(congestion) - logistical ────────────────
        final = round(base_score - c_penalty - logistical_capped, 2)

        results.append({
            "region":             row["region"],
            "tfidf_score":        row["tfidf_score"],
            "knn_profile_score":  row["knn_profile_score"],
            "congestion_index":   round(c_index, 2),
            "congestion_flag":    c_flag,
            "dynamic_penalty":    d_penalty,
            "distance_penalty":   dist_penalty,
            "commute_minutes":    round(commute),
            "total_penalty":      logistical_capped,
            "final_match_score":  final,
        })

    if not results:
        return pd.DataFrame(columns=_OUTPUT_COLS)

    return (
        pd.DataFrame(results)
        .sort_values("final_match_score", ascending=False)
        .reset_index(drop=True)
    )


print("Recommendation engine loaded (TF-IDF + KNN + Ψ + spatial cost + ethical filter + temporal).")


Recommendation engine loaded (TF-IDF + KNN + Ψ + spatial cost + ethical filter + temporal).


## Checking TF-IDF Stability

TF-IDF scores are sensitive to corpus composition. With a small survivor set, dropping one destination can shift the normalised scores significantly. This cell runs a bootstrap test: 100 iterations of random 80% subsamples, recomputing TF-IDF each time, then reporting the mean, standard deviation, and coefficient of variation per destination. High CV means the destination's ranking is fragile and depends on who else is in the pool. This is a useful diagnostic for understanding how much to trust the TF-IDF component of the score.


In [20]:
# ── Upgrade 6: Bootstrap TF-IDF Stability ────────────────────────────────────

def bootstrap_tfidf_stability(df, anchor, n_bootstrap=100, sample_frac=0.80):
    """
    Run TF-IDF scoring N times on random 80% subsamples of the corpus.
    Returns DataFrame with: region, tfidf_mean, tfidf_std, tfidf_cv.
    
    Rationale: With small survivor sets, min-max normalisation is highly
    sensitive to corpus composition. A destination scoring 14.56 raw may
    normalise to 0 or 85 depending on who else survived Stage 1.
    """
    np.random.seed(42)
    orbits = df[df["region"] != anchor].copy()
    all_scores = {r: [] for r in orbits["region"].values}
    
    for _ in range(n_bootstrap):
        # Random 80% subsample of orbits + always include anchor
        sample_orbits = orbits.sample(frac=sample_frac)
        sample_df = pd.concat([df[df["region"] == anchor], sample_orbits]).reset_index(drop=True)
        
        # Compute raw TF-IDF scores
        tag_strings = sample_df["tags"].apply(lambda t: " ".join(t))
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(tag_strings)
        anchor_idx = sample_df.index[sample_df["region"] == anchor][0]
        cos_sim = cosine_similarity(tfidf_matrix[anchor_idx], tfidf_matrix).flatten()
        
        for i, region in enumerate(sample_df["region"].values):
            if region != anchor and region in all_scores:
                all_scores[region].append(cos_sim[i] * 100)
    
    # Compute statistics per destination
    rows = []
    for region, scores in all_scores.items():
        if len(scores) > 0:
            mean_s = np.mean(scores)
            std_s = np.std(scores)
            cv_s = std_s / mean_s if mean_s > 0 else 0
            rows.append({
                "region": region,
                "tfidf_mean": round(mean_s, 2),
                "tfidf_std": round(std_s, 2),
                "tfidf_cv": round(cv_s, 4),
            })
    
    result = pd.DataFrame(rows).sort_values("tfidf_cv", ascending=False).reset_index(drop=True)
    return result


print("Bootstrap TF-IDF stability function loaded.")


Bootstrap TF-IDF stability function loaded.


## Tourist Acceptance Model

Getting a high recommendation score does not mean the tourist will actually go. This cell models acceptance as a logistic sigmoid: P(accept) = 1 / (1 + exp(-k * (score - threshold))). The `k` parameter controls steepness (higher k = sharper accept/reject boundary), and the threshold is the tourist's reservation utility, the minimum score they would consider. Scores well above threshold are near-certain accepts; scores well below are near-certain rejects. This feeds into the multi-round simulation where each tourist independently decides whether to divert based on their acceptance probability.


In [21]:
# ── Upgrade 3: Behavioral Acceptance Model with Reservation Utility ───────────

def acceptance_probability(f_alt, u_reservation, k=0.1):
    """
    Logistic acceptance function:
      P(accept) = σ(k · (F_alt − U_reservation))
    where σ(x) = 1 / (1 + e^(-x))
    
    Parameters
    ----------
    f_alt : float — final_match_score of the alternative destination
    u_reservation : float — reservation utility threshold (minimum score
        below which tourist will not divert regardless of recommendation)
    k : float — sensitivity parameter. Higher k → sharper accept/reject
        boundaries. Lower k → more uniform acceptance across score ranges.
    
    Assumptions (documented per spec):
    - Tourists are modelled as independent decision-makers.
    - Acceptance probability is stationary within a round but recalculated
      each round as U_reservation updates with the evolving score distribution.
    - Social influence and group herding effects are not modelled (future work).
    """
    # Logistic sigmoid: σ(k · (score - threshold))
    x = k * (f_alt - u_reservation)
    return 1.0 / (1.0 + np.exp(-x))


print("Behavioral acceptance model loaded (logistic with reservation utility).")


Behavioral acceptance model loaded (logistic with reservation utility).


## Multi-Round Redistribution Simulation

One round of recommendations changes the congestion landscape: the anchor gets lighter, some orbits get heavier. This cell simulates multiple rounds of that feedback loop. It adds n_tourists to the anchor, generates recommendations on the current state, probabilistically redistributes tourists using the acceptance model (binomial sampling per destination), updates the counts, and repeats. It tracks anchor load, load standard deviation, and Shannon entropy per round. The simulation always operates on df.copy() so the original dataset stays clean.


In [22]:
# ── Upgrade 2: Redistribution Simulation with Load Decay ─────────────────────

def compute_redistribution_metrics(load_before, load_after):
    """
    Compute before/after redistribution metrics.
    Returns dict with std_before, std_after, max_reduction, entropy_before, entropy_after.
    """
    def entropy(loads):
        total = loads.sum()
        if total == 0:
            return 0.0
        p = loads / total
        p = p[p > 0]  # avoid log(0)
        return -np.sum(p * np.log(p))
    
    return {
        "std_before": round(load_before.std(), 2),
        "std_after": round(load_after.std(), 2),
        "entropy_before": round(entropy(load_before), 4),
        "entropy_after": round(entropy(load_after), 4),
    }


def simulate_with_decay(df, anchor, n_tourists=1000, n_rounds=5,
                         group_size=16, tourist_type="international",
                         k=0.1, reservation_utility=None):
    """
    Multi-round redistribution simulation with load decay.
    
    1. Simulate n_tourists initially selecting the anchor destination.
    2. Apply recommendation model to generate ranked alternatives.
    3. Redistribute probabilistically using behavioral acceptance model.
    4. After each round, UPDATE tourist_counts to reflect redirected tourists.
    5. Recalculate congestion indices using updated counts.
    6. Repeat for n_rounds.
    """
    np.random.seed(42)
    
    # ALWAYS operate on a copy — never mutate the original df
    df_sim = df.copy()
    
    # Track load at anchor per round
    round_metrics = []
    
    # Record initial state (round 0)
    initial_loads = df_sim["tourist_count"].copy()
    initial_entropy_vals = initial_loads / initial_loads.sum()
    initial_entropy_vals = initial_entropy_vals[initial_entropy_vals > 0]
    initial_entropy = -np.sum(initial_entropy_vals * np.log(initial_entropy_vals))
    
    round_metrics.append({
        "round": 0,
        "anchor_load": df_sim.loc[df_sim["region"] == anchor, "tourist_count"].values[0],
        "load_std": round(df_sim["tourist_count"].std(), 2),
        "entropy": round(initial_entropy, 4),
        "tourists_redirected": 0,
    })
    
    # Add initial tourists to anchor
    df_sim.loc[df_sim["region"] == anchor, "tourist_count"] += n_tourists
    
    remaining_tourists = n_tourists
    
    for rnd in range(1, n_rounds + 1):
        if remaining_tourists <= 0:
            break
        
        # Generate recommendations using CURRENT simulation state
        results = generate_recommendations(df_sim, anchor, group_size, tourist_type)
        
        if results.empty:
            break
        
        # Compute reservation utility: 40th percentile of current scores
        if reservation_utility is None:
            u_res = np.percentile(results["final_match_score"], 40)
        else:
            u_res = reservation_utility
        
        # Compute acceptance probability per destination
        results["accept_prob"] = results["final_match_score"].apply(
            lambda s: acceptance_probability(s, u_res, k=k)
        )
        
        # Simulate tourist decisions using binomial sampling
        redirected_this_round = 0
        tourists_per_dest = remaining_tourists // len(results)
        
        for _, rec_row in results.iterrows():
            prob = rec_row["accept_prob"]
            # Each tourist independently decides
            accepted = np.random.binomial(tourists_per_dest, prob)
            
            if accepted > 0:
                # Move tourists from anchor to this destination
                df_sim.loc[df_sim["region"] == rec_row["region"], "tourist_count"] += accepted
                df_sim.loc[df_sim["region"] == anchor, "tourist_count"] -= accepted
                redirected_this_round += accepted
        
        remaining_tourists -= redirected_this_round
        
        # Compute round metrics with updated loads
        loads = df_sim["tourist_count"].astype(float)
        p = loads / loads.sum()
        p = p[p > 0]
        entropy_val = -np.sum(p * np.log(p))
        
        round_metrics.append({
            "round": rnd,
            "anchor_load": df_sim.loc[df_sim["region"] == anchor, "tourist_count"].values[0],
            "load_std": round(loads.std(), 2),
            "entropy": round(entropy_val, 4),
            "tourists_redirected": redirected_this_round,
        })
    
    metrics_df = pd.DataFrame(round_metrics)
    
    # Summary
    final_metrics = compute_redistribution_metrics(initial_loads, df_sim["tourist_count"])
    
    print("\n" + "─" * 70)
    print("REDISTRIBUTION SIMULATION RESULTS")
    print("─" * 70)
    print(f"Anchor: {anchor} | Tourists: {n_tourists} | Rounds: {n_rounds}")
    print(f"Load Std Before: {final_metrics['std_before']}  →  After: {final_metrics['std_after']}")
    print(f"Entropy Before:  {final_metrics['entropy_before']}  →  After: {final_metrics['entropy_after']}")
    anchor_before = initial_loads[df["region"] == anchor].values[0]
    anchor_after = df_sim.loc[df_sim["region"] == anchor, "tourist_count"].values[0]
    print(f"Max Anchor Load Reduction: {anchor_before} → {anchor_after} ({anchor_before - anchor_after} tourists)")
    print("\nPer-Round Summary:")
    print(metrics_df.to_string(index=False))
    print("─" * 70)
    
    return metrics_df, df_sim


print("Redistribution simulation with load decay loaded.")


Redistribution simulation with load decay loaded.


## Behavioural Parameter Calibration (Phase 3.5)

The acceptance model has two free parameters: `k` (steepness) and the reservation utility percentile. This cell does a grid search over k=[0.05, 0.10, 0.20, 0.30] and percentile=[30, 40, 50], running the full multi-round simulation for each combination with stdout suppressed via `contextlib.redirect_stdout`. It outputs a styled table sorted by total_redirected, letting you see which parameter combination produces the most effective redistribution. Note that this sweep uses the static engine (no travel_month) since it is calibrating the behavioural model independently of temporal effects.


In [23]:
# ── Phase 3.5: Behavioral Calibration Sweep ──────────────────────────────────
# Sweeps k (sensitivity) × percentile (reservation utility) combinations
# to find optimal behavioral model parameters for redistribution.
# Uses the STATIC engine (no travel_month) — calibrates behavioral model only,
# independent of temporal adjustments.


def run_calibration_sweep(df, anchor, n_tourists=1000, rounds=5,
                          group_size=16, tourist_type="international"):
    """
    Sweep k and reservation-utility percentile to find optimal behavioral parameters.
    
    For each (k, percentile) combination:
      1. Generate recommendations (static engine, no temporal adjustment)
      2. Compute reservation utility from score distribution
      3. Run simulation with pre-computed utility (stdout suppressed)
      4. Extract metrics from returned metrics_df
    
    Returns styled DataFrame sorted by total_redirected DESC.
    """
    # ── Sweep parameters ─────────────────────────────────────────────────────
    k_values = [0.05, 0.10, 0.20, 0.30]
    percentiles = [30, 40, 50]

    sweep_results = []

    # Step 1 — Generate score distribution once (static engine)
    results = generate_recommendations(df, anchor, group_size, tourist_type)

    if results.empty:
        print("⚠ No recommendations generated — cannot run calibration sweep.")
        return pd.DataFrame()

    for k in k_values:
        for percentile in percentiles:
            # Step 2 — Compute reservation utility FROM score distribution
            u_res = np.percentile(results["final_match_score"], percentile)

            # Step 3 — Run simulation with stdout suppressed
            # Seed is set internally by simulate_with_decay (np.random.seed(42)).
            # No external seed reset needed — the simulation controls its own RNG state.
            with contextlib.redirect_stdout(io.StringIO()):
                metrics_df, _ = simulate_with_decay(
                    df, anchor, n_tourists, rounds, group_size,
                    tourist_type, k=k, reservation_utility=u_res
                )

            # Step 4 — Extract metrics
            total_redirected = metrics_df["tourists_redirected"].sum()
            per_round = metrics_df["tourists_redirected"].tolist()
            anchor_before = metrics_df.loc[0, "anchor_load"]
            anchor_after = metrics_df.iloc[-1]["anchor_load"]
            entropy_after = metrics_df.iloc[-1]["entropy"]

            sweep_results.append({
                "k": k,
                "percentile": percentile,
                "total_redirected": total_redirected,
                "per_round": per_round,
                "anchor_before": anchor_before,
                "anchor_after": anchor_after,
                "entropy_after": round(entropy_after, 4),
            })

    sweep_df = (
        pd.DataFrame(sweep_results)
        .sort_values("total_redirected", ascending=False)
        .reset_index(drop=True)
    )

    # Display styled table
    styled = (
        sweep_df.style
        .background_gradient(subset=["total_redirected"], cmap="RdYlGn")
        .set_caption(
            f"Phase 3.5 Calibration Sweep | Anchor: {anchor} | "
            f"n={n_tourists} | rounds={rounds}"
        )
    )
    print("\n" + "═" * 70)
    print("PHASE 3.5: BEHAVIORAL CALIBRATION SWEEP")
    print("═" * 70)
    display(styled)

    return sweep_df


print("Phase 3.5: Behavioral Calibration Sweep loaded.")


Phase 3.5: Behavioral Calibration Sweep loaded.


## Validation Framework

Four layers of automated tests that run before you trust any output. Layer 1 (sanity checks): no negative scores, no scores above theoretical max, anchor excluded from results, meaningful score spread. Layer 2 (benchmarks): overtourism deterrence ordering, score floor on filtered output, coach penalty direction, small-group exemption. Layer 3 (scale anchors): prints the top score as a percentage of the theoretical ceiling to give context for what the numbers mean. Layer 4 (direction tests): verifies that distance penalty, congestion deterrence, and capacity-adjusted spatial cost all push scores in the correct direction. If any test fails, something is structurally wrong.


In [24]:
# ── Upgrade 8: Model Validation Framework ────────────────────────────────────
# Validation must run before result interpretation.
# Passing all four layers is the minimum bar for trusting output.
# A result that looks reasonable but fails a direction test is wrong.


def _generate_unfiltered(df_src, anchor, group_size, tourist_type):
    """Generate recommendations WITHOUT ethical constraints for validation.
    Direction tests and benchmarks need to compare scoring logic across all
    congestion flags, including destinations ethical constraints remove."""
    tourist_type = tourist_type.lower().strip()
    anchor_row = df_src.loc[df_src["region"] == anchor]
    if anchor_row.empty:
        return pd.DataFrame()
    transit_graph = build_japan_astar_graph(df_src)
    tfidf_scores = _compute_tfidf_scores(df_src, anchor)
    orbits = df_src[df_src["region"] != anchor].copy()
    orbits["tfidf_score"] = orbits["region"].map(tfidf_scores)
    survivors = orbits[orbits["tfidf_score"] > 0.0].copy()
    if survivors.empty:
        return pd.DataFrame()
    tmin = survivors["tfidf_score"].min()
    tmax = survivors["tfidf_score"].max()
    if tmax > tmin:
        survivors["tfidf_score"] = ((survivors["tfidf_score"] - tmin) / (tmax - tmin) * 100).round(2)
    else:
        survivors["tfidf_score"] = 75.0
    survivor_list = survivors["region"].tolist()
    knn_scores = _compute_knn_profile_scores(df_src, anchor, survivor_list)
    survivors["knn_profile_score"] = survivors["region"].map(knn_scores)
    results = []
    for _, row in survivors.iterrows():
        commute = get_astar_commute_minutes(transit_graph, anchor, row["region"])
        if tourist_type == "international":
            attractiveness = (row["cultural_score"]*0.40 + row["cleanliness_score"]*0.30 + row["experiential_score"]*0.20 + row["infrastructure_score"]*0.10)
        else:
            attractiveness = (row["experiential_score"]*0.40 + row["infrastructure_score"]*0.30 + row["cleanliness_score"]*0.20 + row["cultural_score"]*0.10)
        base_score = row["tfidf_score"]*0.50 + row["knn_profile_score"]*0.30 + attractiveness*0.20
        load_ratio = row["tourist_count"] / row["capacity"]
        load_score = piecewise_load_score(load_ratio)
        delay_score = min((row["avg_delay_minutes"] / 45) * 100, 100)
        c_index = compute_congestion_index(load_score, delay_score)
        c_flag = congestion_flag(c_index)
        c_penalty = congestion_penalty(c_index)
        d_penalty = compute_dynamic_penalty(row, group_size)
        safe_load_ratio = min(load_ratio, 0.95)
        dist_penalty = round(min(SPATIAL_COST_WEIGHT * commute / (1 - safe_load_ratio), 20), 2)
        logistical_raw = round(d_penalty + dist_penalty, 2)
        logistical_capped = round(min(logistical_raw, 0.40 * base_score), 2)
        final = round(base_score - c_penalty - logistical_capped, 2)
        results.append({"region": row["region"], "tfidf_score": row["tfidf_score"],
            "knn_profile_score": row["knn_profile_score"], "congestion_index": round(c_index, 2),
            "congestion_flag": c_flag, "dynamic_penalty": d_penalty,
            "distance_penalty": dist_penalty, "commute_minutes": round(commute),
            "total_penalty": logistical_capped, "final_match_score": final,
            "load_ratio": round(load_ratio, 3)})
    if not results:
        return pd.DataFrame()
    return pd.DataFrame(results).sort_values("final_match_score", ascending=False).reset_index(drop=True)


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 1 — SANITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════
def run_sanity_checks(results_df, anchor_region):
    """Run 5 sanity checks on the FILTERED recommendation output."""
    all_pass = True
    print("\n── LAYER 1: SANITY CHECKS ──────────────────────────────────────")
    
    if (results_df["final_match_score"] < 0).any():
        print("  ✗ FAIL: Negative scores detected.")
        all_pass = False
    else:
        print("  ✓ PASS: No negative scores")
    
    if (results_df["final_match_score"] > 115).any():
        print("  ✗ FAIL: Score exceeds theoretical maximum (115).")
        all_pass = False
    else:
        print("  ✓ PASS: Score ceiling respected")
    
    severely_over = results_df[results_df["congestion_flag"] == "Severely overtouristed"]
    under = results_df[results_df["congestion_flag"].isin(["Undertouristed", "Severely undertouristed"])]
    if not severely_over.empty and not under.empty:
        if severely_over["final_match_score"].max() > under["final_match_score"].max():
            print("  ✗ FAIL: Overtouristed outranks undertouristed.")
            all_pass = False
        else:
            print("  ✓ PASS: Overtourism deterrence ordering correct")
    else:
        print("  ✓ PASS: Overtourism deterrence (post-filter: overtouristed correctly removed)")
    
    if anchor_region in results_df["region"].values:
        print("  ✗ FAIL: Anchor in its own recommendation set.")
        all_pass = False
    else:
        print("  ✓ PASS: Anchor excluded from recommendations")
    
    score_range = results_df["final_match_score"].max() - results_df["final_match_score"].min()
    if score_range < 5:
        print(f"  ⚠ WARN: Score range too narrow ({score_range:.2f}).")
    else:
        print(f"  ✓ PASS: Score spread meaningful ({score_range:.2f})")
    
    return all_pass


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 2 — BENCHMARK SCENARIOS
# ═══════════════════════════════════════════════════════════════════════════════
def run_benchmarks(df_source, generate_fn):
    """Run 4 pre-defined benchmarks using unfiltered results for scoring logic."""
    print("\n── LAYER 2: BENCHMARK SCENARIOS ────────────────────────────────")
    passed = 0
    
    # Benchmark 1 — Overtourism deterrence (unfiltered)
    r = _generate_unfiltered(df_source, "Kyoto", 16, "international")
    under_max = r[r["congestion_flag"].isin(["Undertouristed", "Severely undertouristed"])]["final_match_score"].max()
    over_max = r[r["congestion_flag"] == "Severely overtouristed"]["final_match_score"].max()
    if pd.notna(under_max) and pd.notna(over_max) and under_max > over_max:
        print(f"  ✓ PASS: Benchmark 1 — Overtourism deterrence (under max={under_max:.2f} > over max={over_max:.2f})")
        passed += 1
    else:
        print(f"  ✗ FAIL: Benchmark 1 — under={under_max}, over={over_max}")
    
    # Benchmark 2 — Score floor on FILTERED results
    # Unfiltered results correctly produce negative scores for severely
    # overtouristed destinations (Ψ > base_score). Ethical constraints
    # remove these. We check the filtered output has no negatives.
    r_filtered = generate_fn(df_source, "Kyoto", 16, "international")
    if (r_filtered["final_match_score"] >= 0).all():
        print("  ✓ PASS: Benchmark 2 — Score floor (no negatives in filtered output)")
        passed += 1
    else:
        print("  ✗ FAIL: Benchmark 2 — Negative scores in filtered output")
    
    # Benchmark 3 — Coach penalty direction (group=20, unfiltered)
    r3 = _generate_unfiltered(df_source, "Kyoto", 20, "international")
    penalised = r3[r3["dynamic_penalty"] > 0]["final_match_score"].mean()
    unpenalised = r3[r3["dynamic_penalty"] == 0]["final_match_score"].mean()
    if pd.notna(penalised) and pd.notna(unpenalised) and penalised < unpenalised:
        print(f"  ✓ PASS: Benchmark 3 — Coach penalty reduces scores (pen={penalised:.2f} < unpen={unpenalised:.2f})")
        passed += 1
    else:
        print(f"  ✗ FAIL: Benchmark 3 — pen={penalised}, unpen={unpenalised}")
    
    # Benchmark 4 — Small group penalty exemption (group=5)
    r4 = _generate_unfiltered(df_source, "Kyoto", 5, "international")
    if (r4["dynamic_penalty"] <= 5).all():
        print("  ✓ PASS: Benchmark 4 — Small group penalty exemption")
        passed += 1
    else:
        print("  ✗ FAIL: Benchmark 4 — Large-group penalties on small group")
    
    print(f"\n  Result: {passed}/4 benchmarks passed.")


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 3 — SCALE ANCHORS
# ═══════════════════════════════════════════════════════════════════════════════
def compute_scale_anchors(results_df, tourist_type):
    """Print score context."""
    print("\n── LAYER 3: SCALE ANCHORS ──────────────────────────────────────")
    ceiling = 115
    top_score = results_df["final_match_score"].max()
    top_region = results_df.loc[results_df["final_match_score"].idxmax(), "region"]
    pct_ceiling = round((top_score / ceiling) * 100, 1)
    median_score = results_df["final_match_score"].median()
    lowest_score = results_df["final_match_score"].min()
    print(f"  Theoretical ceiling: {ceiling}")
    print(f"  Top score: {top_score:.2f} ({top_region}) = {pct_ceiling}% of ceiling")
    print(f"  Median score: {median_score:.2f}")
    print(f"  Lowest score: {lowest_score:.2f}")
    print(f"  Interpretation: A score of {top_score:.2f} means {top_region} captures ")
    print(f"  {pct_ceiling}% of what a perfect match would look like under current congestion conditions.")


# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 4 — DIRECTION TESTS (unfiltered for full comparison)
# ═══════════════════════════════════════════════════════════════════════════════
def test_feature_direction(df_source, generate_fn):
    """Test that features change scores in the correct direction."""
    print("\n── LAYER 4: DIRECTION TESTS ────────────────────────────────────")
    
    r = _generate_unfiltered(df_source, "Kyoto", 5, "international")
    
    # Direction Test 1 — Distance penalty direction
    # Compare TWO undertouristed destinations with different commute times
    # to test pure distance ordering when load_ratio is similar.
    # Kanazawa (~130min, undertouristed) vs Okayama (~145min, undertouristed)
    # Both are undertouristed so capacity adjustment is similar.
    # We pick two undertouristed destinations to isolate the distance effect.
    under_dests = r[r["congestion_flag"].isin(["Undertouristed", "Severely undertouristed"])].copy()
    under_dests = under_dests.sort_values("commute_minutes")
    if len(under_dests) >= 2:
        near = under_dests.iloc[0]
        far = under_dests.iloc[-1]
        if far["distance_penalty"] >= near["distance_penalty"]:
            print(f'  ✓ PASS: Distance direction — {far["region"]} ({far["commute_minutes"]}min, dp={far["distance_penalty"]}) >= {near["region"]} ({near["commute_minutes"]}min, dp={near["distance_penalty"]})')
        else:
            print(f'  ✗ FAIL: Distance inverted — {far["region"]} ({far["commute_minutes"]}min, dp={far["distance_penalty"]}) < {near["region"]} ({near["commute_minutes"]}min, dp={near["distance_penalty"]})')
    else:
        print("  ⚠ SKIP: Direction Test 1 — insufficient undertouristed destinations")
    
    # Direction Test 2 — Congestion deterrence direction
    under_scores = r[r["congestion_flag"].isin(["Undertouristed", "Severely undertouristed"])]["final_match_score"]
    over_scores = r[r["congestion_flag"] == "Severely overtouristed"]["final_match_score"]
    if not under_scores.empty and not over_scores.empty:
        under_mean = under_scores.mean()
        over_mean = over_scores.mean()
        if under_mean > over_mean:
            print(f"  ✓ PASS: Congestion deterrence — under mean ({under_mean:.2f}) > over mean ({over_mean:.2f})")
        else:
            print(f"  ✗ FAIL: Congestion inverted — under ({under_mean:.2f}) vs over ({over_mean:.2f})")
    else:
        print("  ⚠ SKIP: Direction Test 2 — insufficient data")
    
    # Direction Test 3 — Capacity-adjusted distance cooperation
    # Far undertouristed should have LOWER distance_penalty than
    # close overtouristed. This confirms capacity adjustment cooperates
    # with congestion — overtouristed destinations get higher spatial
    # friction even when physically closer.
    under_far = r[(r["congestion_flag"].isin(["Undertouristed", "Severely undertouristed"])) &
                  (r["commute_minutes"] > 100)]
    over_close = r[(r["congestion_flag"].isin(["Severely overtouristed", "Approaching overtourism"])) &
                   (r["commute_minutes"] <= 200)]
    if not under_far.empty and not over_close.empty:
        uf = under_far.iloc[0]
        oc = over_close.iloc[0]
        if uf["distance_penalty"] < oc["distance_penalty"]:
            print(f'  ✓ PASS: Spatial-congestion cooperation — {uf["region"]} (far under, {uf["commute_minutes"]}min, dp={uf["distance_penalty"]}) < {oc["region"]} (close over, {oc["commute_minutes"]}min, dp={oc["distance_penalty"]})')
        else:
            print(f'  ✗ FAIL: Spatial not cooperating — {uf["region"]} (dp={uf["distance_penalty"]}) vs {oc["region"]} (dp={oc["distance_penalty"]})')
    else:
        print("  ⚠ SKIP: Direction Test 3 — no matching far-under / close-over pair")


print("Validation framework loaded (4 layers).")


Validation framework loaded (4 layers).


## Scenario Demonstrations

Five worked examples that exercise the engine under different conditions. Scenarios 1-4 test combinations of anchor city, group size, and tourist type (international vs native). Scenario 5 is a targeted verification of the Phase 4 festival logic: it calls `apply_temporal_adjustments` directly on Kyoto during Gion Matsuri (July) and checks that a cultural tourist (tag overlap >= 40%) gets the attractiveness boost with no congestion delta, while a non-cultural tourist (0% overlap) gets the congestion delta with no boost. Expected values are hardcoded and the test reports PASS/FAIL.


In [25]:
# Validation must run before result interpretation.
# Passing all four layers is the minimum bar for trusting output.
# A result that looks reasonable but fails a direction test is wrong.
# Execution order: sanity → direction → benchmarks → scale anchors.

from IPython.display import HTML

# ── Scenario 1: Kyoto | group=16 | international ─────────────────────────────
print("═" * 80)
print("SCENARIO 1: Kyoto | Coach (n=16) | International")
print("═" * 80)
s1 = generate_recommendations(df, "Kyoto", 16, "international")
if not s1.empty:
    diag1 = s1[["region", "tfidf_score", "knn_profile_score",
               "congestion_flag", "dynamic_penalty", "distance_penalty",
               "commute_minutes", "total_penalty", "final_match_score"]]
    display(diag1.style
           .format({"tfidf_score": "{:.2f}", "knn_profile_score": "{:.2f}",
                    "distance_penalty": "{:.2f}",
                    "total_penalty": "{:.2f}", "final_match_score": "{:.2f}"})
           .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
           .background_gradient(subset=["commute_minutes"], cmap="YlOrRd")
           .set_caption("Scenario 1 — Kyoto | Coach | International"))
    
    # ── LAYER 1: Sanity checks ─────────────────────────────────────────
    run_sanity_checks(s1, "Kyoto")
    
    # ── LAYER 4: Direction tests ───────────────────────────────────────
    test_feature_direction(df, generate_recommendations)
    
    # ── LAYER 2: Benchmarks ────────────────────────────────────────────
    run_benchmarks(df, generate_recommendations)
    
    # ── LAYER 3: Scale anchors ─────────────────────────────────────────
    compute_scale_anchors(s1, "international")

    # ── Upgrade 6: Bootstrap TF-IDF stability ──────────────────────────
    print("\n── BOOTSTRAP TF-IDF STABILITY ──────────────────────────────────")
    bootstrap_df = bootstrap_tfidf_stability(df, "Kyoto")
    unstable = bootstrap_df[bootstrap_df["tfidf_cv"] > 0.30]
    if not unstable.empty:
        print("  ⚠ Unstable TF-IDF destinations (CV > 0.30):")
        for _, u_row in unstable.iterrows():
            print(f"    {u_row['region']}: CV={u_row['tfidf_cv']:.4f} (mean={u_row['tfidf_mean']:.2f}, std={u_row['tfidf_std']:.2f})")
    else:
        print("  ✓ All destinations have stable TF-IDF scores (CV ≤ 0.30)")
    print(bootstrap_df.to_string(index=False))

    # ── Upgrade 2: Redistribution simulation ───────────────────────────
    print("\n── REDISTRIBUTION SIMULATION ───────────────────────────────────")
    sim_metrics, _ = simulate_with_decay(
        df, "Kyoto", n_tourists=1000, n_rounds=5,
        group_size=16, tourist_type="international"
    )
else:
    print("  ✗ No results!")

# ── Scenario 2: Kyoto | group=5 | international ──────────────────────────────
print("\n" + "═" * 80)
print("SCENARIO 2: Kyoto | FIT (n=5) | International")
print("═" * 80)
s2 = generate_recommendations(df, "Kyoto", 5, "international")
if not s2.empty:
    print(f"  ✓ Survivors: {len(s2)}")
    negs = s2[s2["final_match_score"] < 0]
    print(f"  ✓ No negative scores: {len(negs) == 0}")
    display(s2[["region", "tfidf_score", "distance_penalty", "commute_minutes",
               "total_penalty", "final_match_score"]].style
           .format({"tfidf_score": "{:.2f}", "distance_penalty": "{:.2f}",
                    "total_penalty": "{:.2f}", "final_match_score": "{:.2f}"})
           .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
           .set_caption("Scenario 2 — Kyoto | FIT | International"))
    compute_scale_anchors(s2, "international")
else:
    print("  ✗ No results!")

# ── Scenario 3: Tokyo | group=20 | native ────────────────────────────────────
print("\n" + "═" * 80)
print("SCENARIO 3: Tokyo | Coach (n=20) | Native")
print("═" * 80)
s3 = generate_recommendations(df, "Tokyo", 20, "native")
if not s3.empty:
    print(f"  ✓ Survivors: {len(s3)}")
    display(s3[["region", "distance_penalty", "commute_minutes", "total_penalty",
               "final_match_score"]].style
           .format({"distance_penalty": "{:.2f}", "total_penalty": "{:.2f}", "final_match_score": "{:.2f}"})
           .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
           .set_caption("Scenario 3 — Tokyo | Coach | Native"))
    compute_scale_anchors(s3, "native")
else:
    print("  ✗ No results!")

# ── Scenario 4: Nara | group=3 | native ──────────────────────────────────────
print("\n" + "═" * 80)
print("SCENARIO 4: Nara | FIT (n=3) | Native")
print("═" * 80)
s4 = generate_recommendations(df, "Nara", 3, "native")
if not s4.empty:
    print(f"  ✓ Survivors: {len(s4)}")
    print(f"  ✓ No negative scores: {(s4['final_match_score'] >= 0).all()}")
    compute_scale_anchors(s4, "native")
else:
    print("  ✗ No results!")

# ── Scenario 5: Kyoto | July | Phase 4 Festival Logic Verification ────────
# The festival logic in apply_temporal_adjustments modifies the ANCHOR row.
# Since the anchor is excluded from its own recommendation output, the
# effect is invisible in the alternatives table. We verify by calling
# apply_temporal_adjustments directly and inspecting Kyoto's event columns.
#
# Expected behavior for Gion Matsuri (July, affected_tags=5):
#   Cultural (3/5 = 60% >= 40%): attractiveness_delta=+20, congestion_delta=0
#   Non-cultural (0/5 = 0% < 40%): attractiveness_delta=0, congestion_delta=+25
print("\n" + "═" * 80)
print("SCENARIO 5: Kyoto | July | Phase 4 Festival Logic Verification")
print("═" * 80)

s5_pass = True
for label, tags, expected_cong, expected_attr in [
    ("Cultural",     ["festival", "traditional", "culture"], 0, 20),
    ("Non-cultural", ["ski", "snow", "winter_sport"],        25, 0),
]:
    df_diag = df.copy()
    df_diag, bl = apply_temporal_adjustments(
        df_diag, travel_month=7, tourist_tags=tags,
        days_ahead=15, anchor_region="Kyoto"
    )
    kyoto = df_diag[df_diag["region"] == "Kyoto"].iloc[0]
    ecg = int(kyoto.get("event_congestion_delta", 0))
    eat = int(kyoto.get("event_attractiveness_delta", 0))

    cong_ok = ecg == expected_cong
    attr_ok = eat == expected_attr
    status = "✓" if (cong_ok and attr_ok) else "✗"

    print(
        f"  {status} {label:14s} profile → "
        f"congestion_delta={ecg:+d} (expect {expected_cong:+d}), "
        f"attractiveness_delta={eat:+d} (expect {expected_attr:+d})"
    )
    if not (cong_ok and attr_ok):
        s5_pass = False

# Verify match ratio computation
gion_tags = ["festival", "traditional", "culture", "parade", "summer"]
cultural_ratio = len(set(["festival", "traditional", "culture"]) & set(gion_tags)) / len(gion_tags)
noncultural_ratio = len(set(["ski", "snow", "winter_sport"]) & set(gion_tags)) / len(gion_tags)
print(f"\n  Match ratios: cultural={cultural_ratio:.0%} (>= 40% threshold), "
      f"non-cultural={noncultural_ratio:.0%} (< 40% threshold)")

if s5_pass:
    print("\n  ✓ PASS: Festival logic correctly distinguishes cultural vs non-cultural profiles.")
    print("    Cultural profile → stay signal (attractiveness boost, no congestion penalty)")
    print("    Non-cultural profile → redirect signal (congestion penalty, no attractiveness boost)")
else:
    print("\n  ✗ FAIL: Festival logic produced unexpected deltas.")

# Note: simulate_with_decay does not currently pass travel_month to
# generate_recommendations, so simulation-level temporal divergence
# testing requires extending the simulation signature (future work).
print("─" * 80)


════════════════════════════════════════════════════════════════════════════════
SCENARIO 1: Kyoto | Coach (n=16) | International
════════════════════════════════════════════════════════════════════════════════
  ⊘ FILTERED: Nara — failed: capacity ceiling (140% > 80%)
  ⊘ FILTERED: Hiroshima — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Kamakura — failed: capacity ceiling (138% > 80%)
  ⊘ FILTERED: Nikko — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Fukuoka — failed: capacity ceiling (108% > 80%)
  ⊘ FILTERED: Shirakawa-go — failed: capacity ceiling (110% > 80%), infrastructure floor (50 < 55)
  ⊘ FILTERED: Shimane — failed: infrastructure floor (50 < 55)
  ⊘ FILTERED: Kurokawa Onsen — failed: infrastructure floor (45 < 55)


,region,tfidf_score,knn_profile_score,congestion_flag,dynamic_penalty,distance_penalty,commute_minutes,total_penalty,final_match_score
0,Kanazawa,100.00,75.12,Undertouristed,0,13.00,130,13.00,84.22
1,Okayama,31.20,71.11,Undertouristed,0,11.60,145,11.60,49.45
2,Matsumoto,30.68,76.93,Undertouristed,0,20.00,300,20.00,43.34
3,Matsuyama,21.25,83.32,Undertouristed,0,20.00,350,20.00,40.24
4,Koyasan,20.88,83.24,Undertouristed,25,10.10,105,21.08,39.63
5,Aomori,9.05,64.13,Severely undertouristed,15,20.00,350,15.73,38.59
6,Kagawa,10.32,79.72,Undertouristed,0,16.67,200,16.67,36.43
7,Nagasaki,6.00,91.62,Undertouristed,0,20.00,280,18.94,36.41
8,Nagano,10.05,83.09,Undertouristed,0,20.00,250,18.80,36.19
9,Sendai,0.00,83.08,Undertouristed,0,20.00,250,16.57,32.85



── LAYER 1: SANITY CHECKS ──────────────────────────────────────
  ✓ PASS: No negative scores
  ✓ PASS: Score ceiling respected
  ✓ PASS: Overtourism deterrence (post-filter: overtouristed correctly removed)
  ✓ PASS: Anchor excluded from recommendations
  ✓ PASS: Score spread meaningful (65.83)

── LAYER 4: DIRECTION TESTS ────────────────────────────────────
  ✓ PASS: Distance direction — Shimane (385min, dp=20.0) >= Koyasan (105min, dp=10.1)
  ✓ PASS: Congestion deterrence — under mean (40.14) > over mean (-40.43)
  ✓ PASS: Spatial-congestion cooperation — Kanazawa (far under, 130min, dp=13.0) < Shirakawa-go (close over, 190min, dp=20.0)

── LAYER 2: BENCHMARK SCENARIOS ────────────────────────────────
  ✓ PASS: Benchmark 1 — Overtourism deterrence (under max=84.22 > over max=-41.50)
  ⊘ FILTERED: Nara — failed: capacity ceiling (140% > 80%)
  ⊘ FILTERED: Hiroshima — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Kamakura — failed: capacity ceiling (138% > 80%)
  ⊘ FILTERED: N

,region,tfidf_score,distance_penalty,commute_minutes,total_penalty,final_match_score
0,Kanazawa,100.00,13.00,130,13.00,84.22
1,Okayama,31.20,11.60,145,11.60,49.45
2,Koyasan,20.88,10.10,105,15.10,45.61
3,Matsumoto,30.68,20.00,300,20.00,43.34
4,Matsuyama,21.25,20.00,350,20.00,40.24
5,Aomori,9.05,20.00,350,15.73,38.59
6,Kagawa,10.32,16.67,200,16.67,36.43
7,Nagasaki,6.00,20.00,280,18.94,36.41
8,Nagano,10.05,20.00,250,18.80,36.19
9,Sendai,0.00,20.00,250,16.57,32.85



── LAYER 3: SCALE ANCHORS ──────────────────────────────────────
  Theoretical ceiling: 115
  Top score: 84.22 (Kanazawa) = 73.2% of ceiling
  Median score: 36.42
  Lowest score: 18.39
  Interpretation: A score of 84.22 means Kanazawa captures 
  73.2% of what a perfect match would look like under current congestion conditions.

════════════════════════════════════════════════════════════════════════════════
SCENARIO 3: Tokyo | Coach (n=20) | Native
════════════════════════════════════════════════════════════════════════════════
  ⊘ FILTERED: Osaka — failed: capacity ceiling (125% > 80%)
  ⊘ FILTERED: Hiroshima — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Sapporo — failed: capacity ceiling (114% > 80%)
  ⊘ FILTERED: Fukuoka — failed: capacity ceiling (108% > 80%)
  ✓ Survivors: 12


,region,distance_penalty,commute_minutes,total_penalty,final_match_score
0,Sendai,8.36,90,8.36,93.91
1,Toyama,20.00,315,18.19,42.30
2,Nagasaki,20.00,440,20.00,41.09
3,Matsumoto,12.25,140,12.25,39.58
4,Tottori,20.00,425,15.64,38.46
5,Beppu,20.00,450,18.30,35.45
6,Kagoshima,20.00,405,18.09,35.13
7,Matsuyama,20.00,510,17.93,34.91
8,Kanazawa,20.00,290,17.42,34.13
9,Kagawa,20.00,360,17.13,33.69



── LAYER 3: SCALE ANCHORS ──────────────────────────────────────
  Theoretical ceiling: 115
  Top score: 93.91 (Sendai) = 81.7% of ceiling
  Median score: 35.29
  Lowest score: 21.37
  Interpretation: A score of 93.91 means Sendai captures 
  81.7% of what a perfect match would look like under current congestion conditions.

════════════════════════════════════════════════════════════════════════════════
SCENARIO 4: Nara | FIT (n=3) | Native
════════════════════════════════════════════════════════════════════════════════
  ⊘ FILTERED: Kyoto — failed: capacity ceiling (150% > 80%)
  ⊘ FILTERED: Hakone — failed: capacity ceiling (150% > 80%)
  ⊘ FILTERED: Hiroshima — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Kamakura — failed: capacity ceiling (138% > 80%)
  ⊘ FILTERED: Nikko — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Sapporo — failed: capacity ceiling (114% > 80%)
  ⊘ FILTERED: Fukuoka — failed: capacity ceiling (108% > 80%)
  ⊘ FILTERED: Shirakawa-go — failed: cap

## Interactive Dashboard

The user-facing output. Dropdown menus for anchor region and tourist type, a slider for group size. Selecting any combination triggers the recommendation engine and renders a styled table with gradient-coloured scores, congestion flags, commute times, and penalty breakdowns. This is where the entire pipeline comes together into something a non-technical user can explore interactively.


In [26]:
# ── Interactive Dashboard (Upgraded Engine) ──────────────────────────────────
out = widgets.Output()

anchor_dropdown = widgets.Dropdown(
    options=sorted(df["region"].unique().tolist()),
    value="Kyoto",
    description="Anchor:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

group_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description="Group Size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

type_dropdown = widgets.Dropdown(
    options=["international", "native"],
    value="international",
    description="Tourist Type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)


def render_recommendations(anchor_region, group_size, tourist_type):
    results = generate_recommendations(
        df=df, anchor_region=anchor_region,
        group_size=group_size, tourist_type=tourist_type,
    )

    DISPLAY = [
        "region", "tfidf_score", "knn_profile_score",
        "congestion_flag", "dynamic_penalty", "distance_penalty",
        "commute_minutes", "total_penalty", "final_match_score",
    ]

    with out:
        clear_output(wait=True)
        if results.empty:
            print("No thematically relevant orbits found (all may have failed ethical constraints).")
            return

        styled = (
            results[DISPLAY].style
            .format({
                "tfidf_score":       "{:.2f}",
                "knn_profile_score": "{:.2f}",
                "distance_penalty":  "{:.2f}",
                "total_penalty":     "{:.2f}",
                "final_match_score": "{:.2f}",
            })
            .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
            .background_gradient(subset=["tfidf_score"],       cmap="Blues")
            .background_gradient(subset=["knn_profile_score"], cmap="PuBuGn")
            .background_gradient(subset=["commute_minutes"],   cmap="YlOrRd")
            .map(
                lambda v: "color: crimson; font-weight: bold"
                    if v == "Severely overtouristed"
                else ("color: darkorange; font-weight: bold"
                    if v == "Approaching overtourism" else ""),
                subset=["congestion_flag"],
            )
            .set_caption(
                f"DSS Recommendations | Anchor: {anchor_region} | "
                f"Group: {group_size} | Type: {tourist_type.capitalize()}"
            )
            .set_table_styles([{"selector": "", "props": [("width", "100%")]}])
        )
        display(HTML(
            '<div style="overflow-x:auto; max-width:100%">'
            + styled.to_html()
            + "</div>"
        ))
        
        # Auto-run scale anchors for dashboard
        compute_scale_anchors(results, tourist_type)


def _on_change(_):
    render_recommendations(
        anchor_dropdown.value,
        group_slider.value,
        type_dropdown.value,
    )


anchor_dropdown.observe(_on_change, names="value")
group_slider.observe(_on_change,    names="value")
type_dropdown.observe(_on_change,   names="value")

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>DSS Dashboard — Upgraded Engine (Ψ + Spatial + Ethical)</h3>"),
    widgets.HBox([anchor_dropdown, type_dropdown]),
    group_slider,
    widgets.HTML(
        "<small style='color:grey'>TF-IDF renormalised within survivors. "
        "Logistical penalty (dynamic + distance) capped at 40% of base score. "
        "Congestion penalty Ψ never capped. "
        "Commute via A* on transit graph, weighted by destination availability (δ=0.05). "
        "Ethical constraints pre-filter destinations. "
        "Group ≥ 15 → coach feasibility penalties.</small>"
    ),
])

display(controls, out)

# Trigger initial render
render_recommendations(
    anchor_dropdown.value,
    group_slider.value,
    type_dropdown.value,
)


Output()

  ⊘ FILTERED: Nara — failed: capacity ceiling (140% > 80%)
  ⊘ FILTERED: Hiroshima — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Kamakura — failed: capacity ceiling (138% > 80%)
  ⊘ FILTERED: Nikko — failed: capacity ceiling (129% > 80%)
  ⊘ FILTERED: Fukuoka — failed: capacity ceiling (108% > 80%)
  ⊘ FILTERED: Shirakawa-go — failed: capacity ceiling (110% > 80%), infrastructure floor (50 < 55)
  ⊘ FILTERED: Shimane — failed: infrastructure floor (50 < 55)
  ⊘ FILTERED: Kurokawa Onsen — failed: infrastructure floor (45 < 55)
